# Stage the shared workshop bucket

Run on an `ml.t3.medium` notebook instance with the `conda_python3` kernel. Run the cells in order. If cell 2 fails with `AccessDenied`, create the bucket and paste the policy through the console (as in the S3 episode) and continue from cell 3.

**Cell 1: names.** The account id and region come from the session.

In [ ]:
import json, boto3, sagemaker

session = sagemaker.Session()
region = session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
s3 = boto3.client("s3", region_name=region)

shared_bucket = "ml-marathon-shared"
roles = ["ml-sagemaker-use", "ml-sagemaker-bedrock-use"]   # execution roles that may read the bucket
release_v = 1                                               # Badger Scribe data release version
repo = "https://github.com/carpentries-incubator/ML_with_AWS_SageMaker"
print(region, account_id, shared_bucket)

**Cell 2: create the bucket, tag it, block public access, attach the read-only policy.**

In [ ]:
try:
    s3.head_bucket(Bucket=shared_bucket)
    print("bucket exists")
except s3.exceptions.ClientError:
    s3.create_bucket(Bucket=shared_bucket, CreateBucketConfiguration={"LocationConstraint": region})
    print("bucket created")

s3.put_bucket_tagging(Bucket=shared_bucket, Tagging={"TagSet": [
    {"Key": "Project", "Value": "Workshop"}, {"Key": "Name", "Value": "Everyone"}, {"Key": "Purpose", "Value": "Storage"}]})
s3.put_public_access_block(Bucket=shared_bucket, PublicAccessBlockConfiguration={
    "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})

policy = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow",
    "Principal": {"AWS": [f"arn:aws:iam::{account_id}:role/{r}" for r in roles]},
    "Action": ["s3:GetObject", "s3:ListBucket"],
    "Resource": [f"arn:aws:s3:::{shared_bucket}", f"arn:aws:s3:::{shared_bucket}/*"]}]}
s3.put_bucket_policy(Bucket=shared_bucket, Policy=json.dumps(policy))
print("read-only policy attached for", roles)

**Cell 3: WattBot inputs** (metadata, questions, PDF corpus built from the metadata). `/tmp` is a small RAM disk on these instances, so the work directory is on the persistent volume; these files are a few hundred MB. The corpus download takes a few minutes.

In [ ]:
work = "/home/ec2-user/SageMaker/stage"
!rm -rf {work} && mkdir -p {work}
%cd {work}
!git clone -q --depth 1 {repo} lesson
!pip install -q pandas requests huggingface_hub
!unzip -q -o lesson/data/data.zip -d data
!mkdir -p wattbot && cp data/data/wattbot/metadata.csv data/data/wattbot/train_QA.csv wattbot/
!python lesson/scripts/build_wattbot_corpus.py --metadata wattbot/metadata.csv --out wattbot/corpus.zip
!aws s3 sync wattbot/ s3://{shared_bucket}/wattbot/

**Cell 4: Badger Scribe data release**, one asset at a time.

In [ ]:
R = f"{repo}/releases/download/badger-scribe-data-v{release_v}"
for a in ["labels", "images-train", "images-test"]:
    !rm -rf bs && mkdir -p bs
    !curl -sSL -o bs/{a}.zip {R}/badger-scribe-v{release_v}-{a}.zip
    !unzip -q -o bs/{a}.zip -d bs/badger-scribe && rm bs/{a}.zip
    !aws s3 sync bs/badger-scribe/ s3://{shared_bucket}/badger-scribe/
!rm -rf bs

**Cell 5: CHURRO-3B weights**, streamed from Hugging Face into S3 file by file. Nothing is written to the instance's disk. About 7.5 GB; ten minutes or so.

In [ ]:
import requests
from huggingface_hub import HfApi, hf_hub_url

model_id = "stanford-oval/churro-3B"
prefix = "badger-scribe/model/churro-3B"
files = [f for f in HfApi().list_repo_files(model_id) if not f.startswith(".")]
for f in files:
    with requests.get(hf_hub_url(model_id, f), stream=True, timeout=60) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        s3.upload_fileobj(r.raw, shared_bucket, f"{prefix}/{f}")   # multipart upload straight from the stream
    print("uploaded", f)

**Cell 6: check**, then clean up.

In [ ]:
!aws s3 ls s3://{shared_bucket}/ --recursive --summarize | tail -2
!aws s3 ls s3://{shared_bucket}/badger-scribe/
!aws s3 ls s3://{shared_bucket}/badger-scribe/model/churro-3B/
!aws s3 ls s3://{shared_bucket}/wattbot/
%cd /home/ec2-user/SageMaker
!rm -rf {work}

Still by hand: run the fine-tuning episode's homework estimator once and copy `train_summary.json`, `eval_before.csv` and `eval_after.csv` from its `model.tar.gz` to `s3://ml-marathon-shared/badger-scribe/instructor/full-run/`.